# 08 · Does connectivity reconfiguration add over simpler baselines?

**Owner:** Jaime · **Status:** sandbox, for discussion (not a decision) · **Dataset:** B (`load_hcp`, 336 subj) · **Date:** 2026-07-21 (rev. after peer review)

The submitted abstract claims load reconfiguration gives *"complementary information beyond static FC."* That comparison was never run before submission. This notebook runs it, and checks a simpler baseline (task activation) that predicts better.

Pipeline reused verbatim from nb `04` (Goutham's `connect_dots` / `get_brain_profile`, 78-network fingerprint, 4 s HRF delay, seed-42 `RidgeCV`). A reproduction gate asserts reconfig r ≈ 0.366 before any new claim.

### Results

| Question | Answer |
|---|---|
| Reconfiguration predicts WM? | Yes. r ≈ **0.366**, external B→A ≈ 0.40 (nb `04`/`05`). |
| Adds over 0-back FC? | No clear gain. ΔR² = **+0.034 ± 0.023** (< 2 sd, heuristic); `0bk + reconfig` (0.333) not better than reconfig alone. |
| Task activation (2bk−0bk) predicts? | Yes, better. r ≈ **0.60** pooled; ≈ **0.48** across held-out people and runs. |
| FC adds over activation? | No clear gain. ΔR² ≈ **−0.003**. |
| Load effect or baseline level? | Cannot separate. Contrast = 2bk−0bk by construction, and per-run centering makes 0bk/2bk anti-correlated (≈ −0.48; contrast vs 0bk ≈ −0.86). One activation axis, three views. |
| Why activation over reconfig FC? | Activation is several times more reliable between runs (≈ 0.17–0.25) than the difference-of-correlations fingerprint (≈ 0.02). |

**Bottom line:** under these representations (360 regional activation values vs a 78-value network-summarized FC fingerprint), activation predicts better than connectivity reconfiguration, which shows no clear incremental gain over single-condition FC. The predictive signal is not specific to connectivity reconfiguration. Same-task dependence applies to every analysis below (see end).

## Step 0 · Setup

Shared A/B layer + Goutham's connectivity functions, verbatim from nb `04`. Same functions → the canonical number is reproducible.

In [1]:
from pathlib import Path
import os, sys, time
import numpy as np
from scipy.stats import pearsonr
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

HERE = Path.cwd(); JAIME = HERE if (HERE/"datasets.py").exists() else HERE/"sandbox"/"jaime"
sys.path.insert(0, str(JAIME))
import datasets as ds, preprocessing as pp
DATA = Path(os.environ.get("GAMMAS_DATA_DIR", JAIME.parents[1]/"data"))
B = ds.spec_b(DATA); DELAY = 4.0
net = np.load(B.task_dir/"regions.npy").T[1]; soc = np.unique(net)

# ----- Goutham's functions, verbatim (guarantees the fingerprint matches nb04) -----
def connect_dots(x): return np.corrcoef(x)
def get_brain_profile(bs, nn, ss):
    fp = []
    for i, ga in enumerate(ss):
        ia = np.where(nn == ga)[0]
        for gb in ss[i:]:
            ib = np.where(nn == gb)[0]
            if len(ia) == 0 or len(ib) == 0: fp.append(0.0); continue
            if ga == gb:
                sub = bs[np.ix_(ia, ia)]; n = sub.shape[0]
                fp.append(sub[np.triu_indices(n, 1)].mean() if n > 1 else 0.0)
            else: fp.append(bs[np.ix_(ia, ib)].mean())
    return fp

crystal = lambda: RidgeCV(alphas=np.logspace(-3, 5, 100))        # team model
def cvp(X, y, seed=42):                                          # leakage-free 5-fold OOF prediction
    pred = np.zeros_like(y, float)
    for tr, te in KFold(5, shuffle=True, random_state=seed).split(X):
        sc = StandardScaler().fit(X[tr]); m = crystal().fit(sc.transform(X[tr]), y[tr])
        pred[te] = m.predict(sc.transform(X[te]))
    return pred
def rep(X, y, seeds=range(20)):                                 # repeated CV -> mean, sd of r
    r = [pearsonr(cvp(X, y, s), y)[0] for s in seeds]; return np.mean(r), np.std(r)
print("ready:", B.name)

ready: Finalist B (339 subj, +resting-state)


## Step 1 · Features

One pass over B, two runs per subject, from the same 4 s-delayed condition frames:
- **FC fingerprint** (78): network-level correlation pattern for 0-back, 2-back, and reconfiguration (2bk−0bk).
- **Activation** (360): mean BOLD per region, per condition.

Kept **pooled** (both runs, main models) and **per-run** (cross-run test). `DVARS` = BOLD-based motion proxy (control).

In [2]:
A0 = {0: [], 1: []}; A2 = {0: [], 1: []}; FPr = {0: [], 1: []}
A0p, A2p, FP0p, FPp, dvars = [], [], [], [], []
subs = ds.load_subjects(B); t0 = time.time()
for k, s in enumerate(subs):
    ts = {r: ds.load_timeseries(B, s, r) for r in (0, 1)}
    fr = {(r, lvl): (lambda f: f[f < ts[r].shape[1]])(pp.condition_frames(B, s, r, lvl, DELAY))
          for r in (0, 1) for lvl in ("0back", "2back")}
    for r in (0, 1):
        A0[r].append(ts[r][:, fr[(r, "0back")]].mean(1)); A2[r].append(ts[r][:, fr[(r, "2back")]].mean(1))
        FPr[r].append(get_brain_profile(connect_dots(ts[r][:, fr[(r, "2back")]]) -
                                        connect_dots(ts[r][:, fr[(r, "0back")]]), net, soc))
    t0f = np.concatenate([ts[0][:, fr[(0, "0back")]], ts[1][:, fr[(1, "0back")]]], 1)
    t2f = np.concatenate([ts[0][:, fr[(0, "2back")]], ts[1][:, fr[(1, "2back")]]], 1)
    A0p.append(t0f.mean(1)); A2p.append(t2f.mean(1))
    FP0p.append(get_brain_profile(connect_dots(t0f), net, soc))                     # 0-back FC fingerprint
    FPp.append(get_brain_profile(connect_dots(t2f) - connect_dots(t0f), net, soc))  # reconfiguration
    dvars.append(np.mean([np.sqrt((np.diff(ts[r], axis=1)**2).mean(0)).mean() for r in (0, 1)]))
A0 = {r: np.array(v) for r, v in A0.items()}; A2 = {r: np.array(v) for r, v in A2.items()}
FPr = {r: np.array(v) for r, v in FPr.items()}
A0p, A2p, FP0p, FPp, dvars = map(np.array, (A0p, A2p, FP0p, FPp, dvars))
beh = pp.behaviour_table(B).set_index("subject").loc[subs]
y = beh["acc_2bk"].to_numpy(float); acc0 = beh["acc_0bk"].to_numpy(float)
act_c = A2p - A0p
print(f"{len(y)} subjects | FC {FPp.shape[1]}-dim | activation {A0p.shape[1]}-dim | {time.time()-t0:.0f}s")

336 subjects | FC 78-dim | activation 360-dim | 5s


## Step 2 · Reproduction gate

Recompute the team's canonical reconfiguration number (r ≈ 0.366). Mismatch → stop. Same pipeline as nb `04`.

In [3]:
g = rep(FPp, y)[0]
print(f"reconfiguration FC repeated-CV r = {g:+.3f}  (nb04 canonical: +0.366)")
assert abs(g - 0.366) < 0.03, "pipeline does not reproduce nb04 -- stop"
print("GATE PASSED")

reconfiguration FC repeated-CV r = +0.366  (nb04 canonical: +0.366)
GATE PASSED


## Step 3 · Model panel

Repeated 5-fold CV, 20 seeds, target `acc_2bk`; only the features change. Activation ≈ 0.60 vs FC ≈ 0.37. 0-back, 2-back and the contrast predict about equally; they are collinear (quantified in the cell below).

In [4]:
panel = [("reconfig FC (78)", FPp), ("activation contrast 2-0 (360)", act_c),
         ("activation 2-back alone (360)", A2p), ("activation 0-back alone (360)", A0p)]
print(f"{'model':32s} {'CV r':>16s}")
for nm, X in panel:
    m, sd = rep(X, y); print(f"{nm:32s} {m:+.3f} +/- {sd:.3f}")
print("\n-> activation ~0.60 vs FC ~0.37; 0-back, 2-back and the contrast predict about equally")
print("   (collinear via per-run centering) -> one activation signal, not specifically a load effect.")

model                                        CV r


reconfig FC (78)                 +0.366 +/- 0.024


activation contrast 2-0 (360)    +0.600 +/- 0.016


activation 2-back alone (360)    +0.569 +/- 0.015


activation 0-back alone (360)    +0.571 +/- 0.014

-> activation ~0.60 vs FC ~0.37; 0-back, 2-back and the contrast predict about equally
   (collinear via per-run centering) -> one activation signal, not specifically a load effect.


In [5]:
# collinearity of the activation features (why 0bk / 2bk / contrast predict alike)
r_0_2 = np.mean([pearsonr(A0p[:, j], A2p[:, j])[0] for j in range(A0p.shape[1])])
r_c_0 = np.mean([pearsonr(act_c[:, j], A0p[:, j])[0] for j in range(A0p.shape[1])])
print(f'mean corr(0-back, 2-back) per ROI = {r_0_2:+.3f}')
print(f'mean corr(contrast, 0-back) per ROI = {r_c_0:+.3f}  (contrast = 2bk - 0bk by construction)')

mean corr(0-back, 2-back) per ROI = -0.482
mean corr(contrast, 0-back) per ROI = -0.855  (contrast = 2bk - 0bk by construction)


## Step 4 · Incremental (nested, same folds)

Fit the simpler feature, then test whether adding the fancier one improves out-of-sample R². ΔR² < 2 sd across seeds is a heuristic, not a formal test: read as "no clear gain", not "adds nothing". Asked twice: reconfig over 0-back FC; FC over activation.

In [6]:
def delta(full, base):                     # incremental R2 on identical folds, per seed (paired)
    d = [r2_score(y, cvp(full, y, s)) - r2_score(y, cvp(base, y, s)) for s in range(20)]
    return np.mean(d), np.std(d)
di, si = delta(np.hstack([FP0p, FPp]), FP0p)   # does reconfiguration add over 0-back FC?
d1, s1 = delta(np.hstack([FPp, act_c]), act_c) # does FC add over activation?
print(f"reconfig FC over 0-back FC : Delta R2 = {di:+.4f} +/- {si:.4f}  [{'adds' if di-2*si>0 else 'no clear gain'}]")
print(f"FC over activation         : Delta R2 = {d1:+.4f} +/- {s1:.4f}  [{'adds' if d1-2*s1>0 else 'no clear gain'}]")

reconfig FC over 0-back FC : Delta R2 = +0.0344 +/- 0.0225  [no clear gain]
FC over activation         : Delta R2 = -0.0030 +/- 0.0065  [no clear gain]


## Step 5 · Controls on the activation contrast

Remove specific alternative explanations: general ability (`acc_0bk`), motion (`DVARS`), permutation (1000, null refit and scored on permuted labels). Surviving these does not prove "no artifact"; only that these controls do not explain the result.

In [7]:
def resid(v, z):
    Zc = np.column_stack([np.ones(len(v)), z]); return v - Zc @ np.linalg.lstsq(Zc, v, rcond=None)[0]
def partial(pred, z): return pearsonr(resid(pred, z), resid(y, z))[0]
def permp(X, obs, n=1000):                       # null: refit on permuted labels, score vs the SAME permuted labels
    rng = np.random.default_rng(0)
    def one(): yp = rng.permutation(y); return pearsonr(cvp(X, yp), yp)[0]
    return (sum(one() >= obs for _ in range(n)) + 1) / (n + 1)
pred = cvp(act_c, y); r = pearsonr(pred, y)[0]
print(f"activation contrast   raw r        = {r:+.3f}")
print(f"  partial | acc_0bk (ability)      = {partial(pred, acc0):+.3f}")
print(f"  partial | DVARS (motion)         = {partial(pred, dvars):+.3f}")
print(f"  partial | acc_0bk + DVARS        = {partial(pred, np.column_stack([acc0, dvars])):+.3f}")
print(f"  permutation p (1000, fixed null) = {permp(act_c, r):.4f}")
print(f"  corr(prediction, DVARS)          = {pearsonr(pred, dvars)[0]:+.3f}  (weak -> not motion)")

activation contrast   raw r        = +0.598
  partial | acc_0bk (ability)      = +0.412
  partial | DVARS (motion)         = +0.580
  partial | acc_0bk + DVARS        = +0.411


  permutation p (1000, fixed null) = 0.0010
  corr(prediction, DVARS)          = -0.190  (weak -> not motion)


## Step 6 · Cross-run: held-out people AND runs

Per fold: train on training subjects' features from one scan, predict held-out subjects' features from the other scan. Holds out both the test subjects and the run. (An earlier version swapped only the run, so every test subject's target was seen → inflated.) Activation ≈ 0.48, connectivity ≈ 0.25.

In [8]:
def cross_run(F0, F1, seed=42):                 # train one run on TRAIN subjects, predict the OTHER run of HELD-OUT subjects
    def one(Fa, Fb):
        pred = np.zeros_like(y, float)
        for tr, te in KFold(5, shuffle=True, random_state=seed).split(Fa):
            sc = StandardScaler().fit(Fa[tr]); m = crystal().fit(sc.transform(Fa[tr]), y[tr])
            pred[te] = m.predict(sc.transform(Fb[te]))   # test subjects held out, features from the other run
        return pearsonr(pred, y)[0]
    return (one(F0, F1) + one(F1, F0)) / 2               # both directions, averaged
print(f"activation contrast cross-run r = {cross_run(A2[0]-A0[0], A2[1]-A0[1]):+.3f}")
print(f"reconfig FC         cross-run r = {cross_run(FPr[0], FPr[1]):+.3f}")

activation contrast cross-run r = +0.475


reconfig FC         cross-run r = +0.246


## Step 7 · Reliability

The reconfiguration fingerprint is a difference of two correlation matrices of very similar conditions → mostly noise (between-run reliability ≈ 0.02). Mean activation is far steadier, hence a higher predictive ceiling.

In [9]:
def rel(F0, F1):
    rr = [pearsonr(F0[:, j], F1[:, j])[0] for j in range(F0.shape[1])]
    return float(np.tanh(np.nanmean(np.arctanh(np.clip(rr, -0.999, 0.999)))))
print(f"activation contrast : {rel(A2[0]-A0[0], A2[1]-A0[1]):+.3f}")
print(f"activation 2-back   : {rel(A2[0], A2[1]):+.3f}")
print(f"reconfig FC         : {rel(FPr[0], FPr[1]):+.3f}")

activation contrast : +0.169
activation 2-back   : +0.249
reconfig FC         : +0.024


## Interpretation (for discussion, not a decision)

**Checked:**
1. Reconfiguration predicts WM (r ≈ 0.37 pooled, replicated; external B→A ≈ 0.40). Stands.
2. No clear incremental gain over single-condition FC (nested ΔR² ≈ +0.03, < 2 sd, heuristic).
3. Task activation predicts better (r ≈ 0.60 pooled; ≈ 0.48 held-out people+runs). Not explained by these controls: partial | DVARS ≈ 0.58, partial | acc_0bk ≈ 0.41, permutation p ≈ 0.001.
4. FC shows no clear gain over activation (ΔR² ≈ −0.003).

**Not a "trait" claim.** The contrast is a linear combination of the two conditions by construction, and per-run centering (`load_timeseries` in [datasets.py](datasets.py)) makes 0-back and 2-back anti-correlated (≈ −0.48; contrast vs 0-back ≈ −0.86). So 0-back, 2-back and the contrast are one activation axis, not independent effects; baseline level cannot be told from load-driven change here.

**Comparison is not fully matched:** 360 regional activation values vs a 78-value network-summarized FC fingerprint. Defensible statement: *under these representations, regional activation predicted better than the network-summarized connectivity fingerprint, and reconfiguration showed no clear incremental gain over 0-back connectivity.*

**Does not invalidate the project.** Connectivity prediction is real and externally validated. This reframes the explanation, not the result.

**Same-task dependence** (all analyses): brain features and behaviour come from the same task, so they share task-state variance. Subject holdout removes leakage, but the result stays associational, not causal.

**Next (21 Jul).** Goutham's committed notebook predicts from the FC fingerprint and does not test activation → no overlap. Keeping the connectivity framing or folding in activation is a group decision.